In [9]:
import TOSICA
import scanpy as sc
import numpy as np
import warnings 
warnings.filterwarnings ("ignore")
import anndata as ad
import pandas as pd
import pickle

In [10]:
import torch
print(torch.__version__)
# print(torch.cuda.get_device_capability(device=None),  torch.cuda.get_device_name(device=None))

1.7.1


In [11]:
ref_adata = sc.read('C:/Users/mz24b548/Documents/GitRepos_local/demo_train.h5ad')
ref_adata = ref_adata[:,ref_adata.var_names]
print(ref_adata)
print(ref_adata.obs.Celltype.value_counts())

View of AnnData object with n_obs × n_vars = 10600 × 3000
    obs: 'Celltype'
    var: 'Gene Symbol'
Celltype
alpha          3136
beta           2966
ductal         1290
acinar         1144
delta           793
PSC             524
PP              356
endothelial     273
macrophage       52
mast             25
epsilon          21
schwann          13
t_cell            7
Name: count, dtype: int64


In [4]:
with open('C:/Users/mz24b548/Documents/GitRepos_local/scLinear_R/Tabula_sapiens_classifier_input/bladder_gene_expression_tabsap_normed','rb') as file:
        gex_normed = pickle.load(file)

with open('C:/Users/mz24b548/Documents/GitRepos_local/scLinear_R/Tabula_sapiens_classifier_input/bladder_tabsap_annotations','rb') as file:
        annotations = pickle.load(file)

In [87]:
gene_names = pd.read_table('C:/Users/mz24b548/Documents/GitRepos_local/scLinear_R/Tabsap_bladder_gene_names.txt', sep = '\t')
gene_names = gene_names['hgnc_symbol']
#Index inherited from R indexing which starts at 1 --> -1 so it starts from 0 so i can match it to AnnData object index
gene_names.index = gene_names.index-1
gene_names.index = gene_names.index.astype(str)

In [44]:
gcounts_tabsap = ad.AnnData(gex_normed).transpose()
gcounts_tabsap.obs["Celltype"] = pd.Categorical(annotations)
gcounts_tabsap.var["Gene Symbol"] = gene_names

In [104]:
gcounts_tabsap_train = gcounts_tabsap[0:5000,:]
gcounts_tabsap_test = gcounts_tabsap[60001:,:]

In [102]:
gcounts_tabsap_train

View of AnnData object with n_obs × n_vars = 10000 × 61759
    obs: 'Celltype'
    var: 'Gene Symbol'

In [105]:
TOSICA.train(gcounts_tabsap_train,gmt_path='human_gobp',label_name='Celltype',epochs=3,project='tabsap_tosica_predicts')

cpu


MemoryError: Unable to allocate 2.30 GiB for an array with shape (5000, 61759) and data type float64